### Load Document

In [20]:
from langchain_community.document_loaders import CSVLoader
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.chat_models.base import init_chat_model
from datetime import datetime
from typing import List
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI




csv_loader = CSVLoader(
    file_path="email.csv",
    encoding="utf-8",
    csv_args={
        'delimiter':",",
        'quotechar':'"'
    }
)

csv_docs=csv_loader.load()
print("\nFirst document:")
print(f"Content:\n{csv_docs[0].page_content}")
print(f"Metadata:\n{csv_docs[0].metadata}")


First document:
Content:
Package Category: Transit
Initial Email: Where is my package? It hasn't moved in 3 days.
Responded Email: Your package is still in transit and expected to update soon.
Metadata:
{'source': 'email.csv', 'row': 0}


In [21]:
csv_docs[0].metadata

{'source': 'email.csv', 'row': 0}

In [22]:
csv_docs[0].page_content

"Package Category: Transit\nInitial Email: Where is my package? It hasn't moved in 3 days.\nResponded Email: Your package is still in transit and expected to update soon."

### Create Chunks

In [23]:

def process_csv(filepath):
    df = pd.read_csv(filepath)
    documents = []
    
    for idx, row in df.iterrows():
        print(row)
        # Create structured content
        content = f"""
        Package Category: {row['Package Category']}
        Initial Email: {row['Initial Email']}
        Responded Email: ${row['Responded Email']}
        """
        
        # Create document with rich metadata
        doc = Document(
            page_content=content,
            metadata={
                'source': filepath,
                'row_index': idx,
                'Package Category': row['Package Category'],
                'Initial Email': row['Initial Email'],
                'Responded Email': row['Responded Email']
            }
        )
        documents.append(doc)
    return documents

In [24]:
chunks = process_csv('email.csv')
chunks

Package Category                                              Transit
Initial Email         Where is my package? It hasn't moved in 3 days.
Responded Email     Your package is still in transit and expected ...
Name: 0, dtype: str
Package Category                                            Delivered
Initial Email       I didn't receive my package even though it say...
Responded Email     We have opened a case to verify the delivery l...
Name: 1, dtype: str
Package Category                                    Shipped
Initial Email                   Has my package shipped yet?
Responded Email     Yes your package shipped earlier today.
Name: 2, dtype: str
Package Category                                        General
Initial Email         I need help updating my delivery address.
Responded Email     Your address has been updated successfully.
Name: 3, dtype: str
Package Category                                       Investigations
Initial Email                   My package seems lost. Can

[Document(metadata={'source': 'email.csv', 'row_index': 0, 'Package Category': 'Transit', 'Initial Email': "Where is my package? It hasn't moved in 3 days.", 'Responded Email': 'Your package is still in transit and expected to update soon.'}, page_content="\n        Package Category: Transit\n        Initial Email: Where is my package? It hasn't moved in 3 days.\n        Responded Email: $Your package is still in transit and expected to update soon.\n        "),
 Document(metadata={'source': 'email.csv', 'row_index': 1, 'Package Category': 'Delivered', 'Initial Email': "I didn't receive my package even though it says delivered.", 'Responded Email': 'We have opened a case to verify the delivery location.'}, page_content="\n        Package Category: Delivered\n        Initial Email: I didn't receive my package even though it says delivered.\n        Responded Email: $We have opened a case to verify the delivery location.\n        "),
 Document(metadata={'source': 'email.csv', 'row_index'

### Vector Store

In [25]:
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [26]:
sampleText = "I lost my package"
embeddings = OpenAIEmbeddings()

embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x00000265E03A4710>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x00000265E03A4150>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [27]:
persist_directory = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(),
    persist_directory=persist_directory,
    collection_name="package_categories"
)


### Test Similarity

In [28]:
query="I cant find my package"


# Search ONLY category first
category_docs = vectorstore.similarity_search(
    f"Category: {query}", 
    k=1
)

category = category_docs[0].metadata["Package Category"]
print("Detected category:", category)

# Search INITIAL email now
similar_initial = vectorstore.similarity_search(
    query,
    k=3,
    filter={"Package Category": category}
)

similar_initial


Detected category: Delivered


[Document(metadata={'row_index': 329, 'Package Category': 'Delivered', 'Responded Email': 'We have contacted the carrier to verify the delivery details.', 'Initial Email': "My tracking says delivered but I can't find the package. Please help.", 'source': 'email.csv'}, page_content="\n        Package Category: Delivered\n        Initial Email: My tracking says delivered but I can't find the package. Please help.\n        Responded Email: $We have contacted the carrier to verify the delivery details.\n        "),
 Document(metadata={'Initial Email': "My tracking says delivered but I can't find the package. Please help.", 'Package Category': 'Delivered', 'Responded Email': 'We have contacted the carrier to verify the delivery details.', 'source': 'email.csv', 'row_index': 329}, page_content="\n        Package Category: Delivered\n        Initial Email: My tracking says delivered but I can't find the package. Please help.\n        Responded Email: $We have contacted the carrier to verify t

### Using LLM

In [29]:
llm=ChatOpenAI(
    model_name="gpt-3.5-turbo"
)

In [30]:
llm = init_chat_model("openai:gpt-3.5-turbo")

In [31]:
retriever=vectorstore.as_retriever(
    search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
)

In [32]:
custom_prompt = ChatPromptTemplate.from_template("""You are an email responder for a package distribution company. Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer. Be sure to greet the user first
                                                 
Context:
{context}

Question: {question}

Answer:""")
custom_prompt


ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an email responder for a package distribution company. Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer. Be sure to greet the user first\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [33]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000265DEFE6390>, search_kwargs={})

In [34]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    { 
        "context":retriever ,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000265DEFE6390>, search_kwargs={}),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an email responder for a package distribution company. Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer. Be sure to greet the user first\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOpenAI(output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tok

In [36]:
response=rag_chain_lcel.invoke("My item is broken. Can I return it?")
response

'Hello,\n\nBased on the information provided, your item has arrived damaged. We are arranging a replacement shipment for you. At this time, there is no need to return the broken item. Please let us know if you need any further assistance. Thank you for your understanding.'

In [37]:
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")
    
    # Get source documents separately if needed
    docs = retriever._get_relevant_documents(question,run_manager=None)
    print("\nSource Documents:")
    for doc in docs:
        print(f"\n--- Source ---")
        print(doc.page_content)

In [39]:
query_rag_lcel("My item is broken. Can I return it?")

Question: My item is broken. Can I return it?
--------------------------------------------------
Answer: Hello,

Based on the provided context, it seems like your item arrived damaged. In this case, we are arranging a replacement shipment for you. You do not need to return the broken item. We appreciate your understanding.

If you have any other questions or concerns, please feel free to let us know.

Thank you.

Source Documents:

--- Source ---

        Package Category: Delivered
        Initial Email: The item arrived damaged. Please help.
        Responded Email: $We are arranging a replacement shipment for you. We appreciate your understanding.
        

--- Source ---

        Package Category: Delivered
        Initial Email: The item arrived damaged. Please help.
        Responded Email: $We are arranging a replacement shipment for you. We appreciate your understanding.
        

--- Source ---

        Package Category: Delivered
        Initial Email: The item arrived damage